

## Manjejo de datos







### Importación de librerías

In [ ]:
import pandas as pd

### CARGA y VERIFICACIÓN DE LOS DATOS


In [16]:
df = pd.read_csv("../../data/processed/corpus_clean.csv")

In [17]:
df.head(15)

,texto,calificacion,lugar,categoria,fuente
0,"Muy bonitas las instalaciones, las aves que an...",5,Hotel Roca Negra del Arenal,alojamiento,booking
1,El lugar es muy bonito con mucho jardín y anim...,5,Hotel Roca Negra del Arenal,alojamiento,booking
2,La calidez del lugar y las personas.,5,Hotel Roca Negra del Arenal,alojamiento,booking
3,El jacuzzi y la piscina. La ubicación y la ate...,5,Hotel Roca Negra del Arenal,alojamiento,booking
4,Todas las zonas impolutas y muy limpias. El pe...,5,Hotel Roca Negra del Arenal,alojamiento,booking
5,"Agradable lugar, muy buena atención de las chi...",5,Hotel Roca Negra del Arenal,alojamiento,booking
6,"El personal, en especial Elisabeth que nos tra...",5,Hotel Roca Negra del Arenal,alojamiento,booking
7,Ubicación e instalaciones La recepcionista que...,5,Hotel Roca Negra del Arenal,alojamiento,booking
8,Muy cómodo Todo positivo,5,Hotel Roca Negra del Arenal,alojamiento,booking
9,Muy bien ambiente natural tiene de todo hast...,5,Hotel Roca Negra del Arenal,alojamiento,booking


### TRADUCCIÓN DE COMENTARIOS AL ESPAÑOL

In [21]:
from langdetect import detect
from langdetect.lang_detect_exception import LangDetectException
from deep_translator import GoogleTranslator

In [22]:
# IDENTIFICA EL IDIOMA DEL COMENTARIO
# SI NO LO PUEDE DETECTAR LO CATEGORIZA COMO 'UNKNOWN'
def idioma_confiable(texto):
    if pd.isna(texto) or len(texto.split()) < 3:
        return "unknown"
    try:
        return detect(texto)
    except LangDetectException:
        return "unknown"

In [23]:
# AGREGA EL IDIOMA DETECTADO EN UNA NUEVA COLUMNA Y HACE UN CONTEO DE LOS IDIOMAS PRESENTES

df["idioma_comentario"] = df["texto"].apply(idioma_confiable)
df["idioma_comentario"].value_counts()

idioma_comentario
en         521
unknown    474
es         178
de         165
fr         138
nl         117
he           9
ca           8
it           7
pt           7
af           4
pl           4
hu           3
no           3
sv           2
lt           2
da           2
vi           1
ro           1
ru           1
el           1
cs           1
ko           1
Name: count, dtype: int64

In [25]:
# FUNCIÓN PARA TRADUCIR EL TEXTO ORIGINAL AL ESPAÑOL
translator = GoogleTranslator(source="auto", target="es")

def traducir_texto(texto, idioma):
    if pd.isna(texto) or texto.strip() == "":
        return texto
    if idioma in ["es", "unknown"]:
        return texto
    try:
        return translator.translate(texto)
    except:
        return texto

In [26]:
# AGREGA EL COMENTARIO YA TRADUCIDO AL CORPUS EN UNA NUEVA COLUMNA
# NO SE MODIFICA EL COMENTARIO ORIGINAL

df["comentarios_espanol"] = df.apply(
    lambda row: traducir_texto(row["texto"], row["idioma_comentario"]), axis=1
)

In [27]:
# SEPARAMOS LOS EMOJIS DEL COMENTARIO

import re

def extraer_emojis(texto):
    if pd.isna(texto):
        return ""
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticonos
        u"\U0001F300-\U0001F5FF"  # símbolos y pictogramas
        u"\U0001F680-\U0001F6FF"  # transporte y mapas
        u"\U0001F1E0-\U0001F1FF"  # banderas
        u"\U00002700-\U000027BF"  # dingbats
        u"\U0001F900-\U0001F9FF"  # símbolos adicionales
        "]+", flags=re.UNICODE)
    return " ".join(emoji_pattern.findall(texto))

def quitar_emojis(texto):
    if pd.isna(texto):
        return texto
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF"
        u"\U00002700-\U000027BF"
        u"\U0001F900-\U0001F9FF"
        "]+", flags=re.UNICODE)
    return emoji_pattern.sub("", texto).strip()


# CADA EMOJI SE COLOCA EN UNA NUEVA COLUMNA

df["emojis"] = df["comentarios_espanol"].apply(extraer_emojis)
df["comentarios_espanol"] = df["comentarios_espanol"].apply(quitar_emojis)

In [28]:
df.head(15)

,texto,calificacion,lugar,categoria,fuente,idioma_comentario,comentarios_espanol,emojis
0,"Muy bonitas las instalaciones, las aves que an...",5,Hotel Roca Negra del Arenal,alojamiento,booking,es,"Muy bonitas las instalaciones, las aves que an...",
1,El lugar es muy bonito con mucho jardín y anim...,5,Hotel Roca Negra del Arenal,alojamiento,booking,es,El lugar es muy bonito con mucho jardín y anim...,
2,La calidez del lugar y las personas.,5,Hotel Roca Negra del Arenal,alojamiento,booking,es,La calidez del lugar y las personas.,
3,El jacuzzi y la piscina. La ubicación y la ate...,5,Hotel Roca Negra del Arenal,alojamiento,booking,es,El jacuzzi y la piscina. La ubicación y la ate...,
4,Todas las zonas impolutas y muy limpias. El pe...,5,Hotel Roca Negra del Arenal,alojamiento,booking,es,Todas las zonas impolutas y muy limpias. El pe...,
5,"Agradable lugar, muy buena atención de las chi...",5,Hotel Roca Negra del Arenal,alojamiento,booking,es,"Agradable lugar, muy buena atención de las chi...",
6,"El personal, en especial Elisabeth que nos tra...",5,Hotel Roca Negra del Arenal,alojamiento,booking,es,"El personal, en especial Elisabeth que nos tra...",
7,Ubicación e instalaciones La recepcionista que...,5,Hotel Roca Negra del Arenal,alojamiento,booking,es,Ubicación e instalaciones La recepcionista que...,
8,Muy cómodo Todo positivo,5,Hotel Roca Negra del Arenal,alojamiento,booking,es,Muy cómodo Todo positivo,
9,Muy bien ambiente natural tiene de todo hast...,5,Hotel Roca Negra del Arenal,alojamiento,booking,es,Muy bien ambiente natural tiene de todo hast...,


In [29]:
# GUARDAMOS EL CORPUS NUEVO EN LA RUTA DETERMINADA

ruta_processed = "../../data/processed/corpus_clean2.csv"

df.to_csv(ruta_processed, index=False)
print("Archivo limpio guardado en data/processed")

Archivo limpio guardado en data/processed


### UNIÓN DE ARCHIVOS LIMPIOS

In [ ]:
# dataset_booking-postagging.csv   +  dataset_Google-Maps-Reviews-Scraper-postagging.csv

RUTA_GOOGLE_MAPS = "../../data/processed/dataset_Google-Maps-Reviews-Scraper-postagging.csv"
RUTA_BOOKING = "../../data/processed/dataset_booking-postagging.csv"
RUTA_SALIDA = "../../data/processed/corpus_final.csv"

COLUMNAS_ORDEN = [
    "lugar", "texto", "calificacion", "categoria", "fuente",
    "idioma_comentario", "comentarios_espanol", "emojis", "penntreebank", "universalpos",
]



In [ ]:
df_google = pd.read_csv(RUTA_GOOGLE_MAPS)
df_booking = pd.read_csv(RUTA_BOOKING)

print(f"Google Maps: {len(df_google)} filas")
print(f"Booking: {len(df_booking)} filas")

In [7]:
filas_antes = len(df_booking)
df_booking = df_booking.dropna(subset=["texto"]).reset_index(drop=True)
print(f"Descartadas {filas_antes - len(df_booking)} filas sin texto ({len(df_booking)} quedan)")

Descartadas 0 filas sin texto (1197 quedan)


In [8]:
faltantes_google = [c for c in COLUMNAS_ORDEN if c not in df_google.columns]
faltantes_booking = [c for c in COLUMNAS_ORDEN if c not in df_booking.columns]
if faltantes_google or faltantes_booking:
    raise ValueError(f"Columnas faltantes -- Google: {faltantes_google} | Booking: {faltantes_booking}")

df_google = df_google[COLUMNAS_ORDEN]
df_booking = df_booking[COLUMNAS_ORDEN]
print("Columnas alineadas correctamente en ambos datasets.")


Columnas alineadas correctamente en ambos datasets.


In [10]:
df_final = pd.concat([df_google, df_booking], ignore_index=True)
print(f"Total filas del corpus final: {len(df_final)}")


Total filas del corpus final: 2402


In [11]:
df_final.to_csv(RUTA_SALIDA, index=False)
print(f"Guardado en: {RUTA_SALIDA}")


Guardado en: ../../data/processed/corpus_final.csv
